In [1]:
pip install pandas mysql-connector-python


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [35]:
import mysql.connector
import pandas as pd
from datetime import datetime

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Root@123#",
    database="HospitalManagement"
)

print("--Database Connected successfully--")


--Database Connected successfully--


In [36]:
class TransparencyLogger:
    @staticmethod
    def log(agent, message):
        print(f"[{agent}] → {message}")


In [37]:
class DataRetrievalAgent:
    def __init__(self, conn):
        self.conn = conn

    def fetch_data(self):
        TransparencyLogger.log("DataRetrievalAgent", "Fetching appointment, doctor, department data")

        query = """
        SELECT 
            a.AppointmentID,
            a.PatientID,
            a.DoctorID,
            a.AppointmentDate,
            d.DepartmentID
        FROM Appointment a
        JOIN Doctor doc ON a.DoctorID = doc.DoctorID
        JOIN Department d ON doc.DepartmentID = d.DepartmentID
        """

        df = pd.read_sql(query, self.conn)

        TransparencyLogger.log(
            "DataRetrievalAgent",
            f"Fetched {len(df)} records with columns {list(df.columns)}"
        )

        return df


In [38]:
class AdmissionCounterAgent:
    def process(self, df):
        TransparencyLogger.log(
            "AdmissionCounterAgent",
            "Counting total appointments per patient"
        )

        grouped = (
            df.groupby("PatientID")
            .size()
            .reset_index(name="TotalAppointments")
        )

        TransparencyLogger.log(
            "AdmissionCounterAgent",
            f"Generated admission counts for {len(grouped)} patients"
        )

        return grouped


In [39]:
class ReadmissionAgent:
    def process(self, admission_df):
        TransparencyLogger.log(
            "ReadmissionAgent",
            "Applying rule: Readmissions = TotalAppointments - 1"
        )

        admission_df["Readmissions"] = admission_df["TotalAppointments"] - 1
        admission_df["Readmissions"] = admission_df["Readmissions"].clip(lower=0)

        for _, row in admission_df.head(5).iterrows():
            TransparencyLogger.log(
                "ReadmissionAgent",
                f"Patient {row.PatientID}: "
                f"{row.TotalAppointments} appointments → "
                f"{row.Readmissions} readmissions"
            )

        return admission_df


In [40]:
class DepartmentAttributionAgent:
    def __init__(self, conn):
        self.conn = conn

    def process(self, readmission_df):
        TransparencyLogger.log(
            "DepartmentAttributionAgent",
            "Mapping patients to department names"
        )

        query = """
        SELECT 
            a.PatientID,
            d.DepartmentID,
            d.DepartmentName
        FROM Appointment a
        JOIN Doctor doc ON a.DoctorID = doc.DoctorID
        JOIN Department d ON doc.DepartmentID = d.DepartmentID
        """

        dept_map = pd.read_sql(query, self.conn)

        merged = pd.merge(dept_map, readmission_df, on="PatientID")

        department_summary = (
            merged.groupby(["DepartmentID", "DepartmentName"])["Readmissions"]
            .sum()
            .reset_index(name="TotalReadmissions")
        )

        return department_summary


In [41]:
class InsightExplanationAgent:
    def process(self, patient_df, department_df):
        TransparencyLogger.log(
            "InsightExplanationAgent",
            "Generating patient and department-level insights"
        )

        # Patient-level insights
        print("\n PATIENT READMISSION SUMMARY\n")
        top_patients = patient_df.sort_values(
            "Readmissions", ascending=False
        ).head(5)
        print(top_patients)

        # Department-level insights
        print("\n🏥 DEPARTMENT READMISSION SUMMARY\n")
        ranked_departments = department_df.sort_values(
            "TotalReadmissions", ascending=False
        )

        for _, row in ranked_departments.iterrows():
            print(
                f"{row.DepartmentName} department handled "
                f"{row.TotalReadmissions} readmissions."
            )



In [42]:
class OrchestratorAgent:
    def __init__(self, conn):
        self.conn = conn

    def run(self):
        TransparencyLogger.log(
            "OrchestratorAgent",
            "Starting swarm intelligence workflow"
        )

        data_agent = DataRetrievalAgent(self.conn)
        raw_data = data_agent.fetch_data()

        admission_agent = AdmissionCounterAgent()
        admission_data = admission_agent.process(raw_data)

        readmission_agent = ReadmissionAgent()
        readmission_data = readmission_agent.process(admission_data)

        department_agent = DepartmentAttributionAgent(self.conn)
        department_data = department_agent.process(readmission_data)

        # DepartmentInsightAgent REMOVED
        # Insights handled here
        insight_agent = InsightExplanationAgent()
        insight_agent.process(readmission_data, department_data)

        TransparencyLogger.log(
            "OrchestratorAgent",
            "Workflow completed successfully"
        )


In [43]:
if __name__ == "__main__":
    orchestrator = OrchestratorAgent(conn)
    orchestrator.run()


[OrchestratorAgent] → Starting swarm intelligence workflow
[DataRetrievalAgent] → Fetching appointment, doctor, department data
[DataRetrievalAgent] → Fetched 1000 records with columns ['AppointmentID', 'PatientID', 'DoctorID', 'AppointmentDate', 'DepartmentID']
[AdmissionCounterAgent] → Counting total appointments per patient
[AdmissionCounterAgent] → Generated admission counts for 628 patients
[ReadmissionAgent] → Applying rule: Readmissions = TotalAppointments - 1
[ReadmissionAgent] → Patient 1: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 2: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 4: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 6: 1 appointments → 0 readmissions
[ReadmissionAgent] → Patient 8: 1 appointments → 0 readmissions
[DepartmentAttributionAgent] → Mapping patients to department names
[InsightExplanationAgent] → Generating patient and department-level insights

 PATIENT READMISSION SUMMARY

     PatientID  TotalAppointment

C:\Users\sonka\AppData\Local\Temp\ipykernel_10144\1300786032.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.conn)
C:\Users\sonka\AppData\Local\Temp\ipykernel_10144\3828183226.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dept_map = pd.read_sql(query, self.conn)


In [45]:
pip install openai


   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ----------------------------- ---------- 0.8/1.1 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 4.7 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- -------------


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
from openai import OpenAI

client = OpenAI(api_key="YOUR_API_KEY")


In [47]:
def intent_detection_agent(user_input):
    medical_keywords = ["fever", "pain", "headache", "cough", "vomit", "dizzy"]
    if any(word in user_input.lower() for word in medical_keywords):
        return "MEDICAL"
    return "DATA"


In [48]:
def medical_llm_agent(symptoms):
    prompt = f"""
You are a medical guidance assistant.
You are NOT a doctor.
Give safe, general advice.

User symptoms: {symptoms}

Rules:
- No diagnosis
- No medicine prescription
- Suggest rest, hydration, observation
- Mention when to visit a hospital
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [49]:
def safety_agent(symptoms):
    danger_signs = ["chest pain", "difficulty breathing", "unconscious", "seizure"]
    if any(sign in symptoms.lower() for sign in danger_signs):
        return "⚠️ Emergency detected. Please go to the nearest hospital immediately."
    return None


In [50]:
def explanation_agent_llm():
    print("\n[Explanation]")
    print("This advice is generated using general medical knowledge.")
    print("It does not replace a doctor consultation.")


In [51]:
def orchestrator():
    print("\nWelcome to Swarm-Intelligent Hospital AI")
    user_input = input("Ask your question: ")

    intent = intent_detection_agent(user_input)

    if intent == "MEDICAL":
        print("\n[Medical Agent Activated]")
        emergency = safety_agent(user_input)
        if emergency:
            print(emergency)
            return

        advice = medical_llm_agent(user_input)
        explanation_agent_llm()
        print("\n[Medical Guidance]")
        print(advice)

    else:
        print("\n[Data Agents Activated]")
        print("Proceeding with hospital analytics workflow...")
        # Existing DB / readmission logic runs here


In [52]:
# ==========================
# SWARM INTELLIGENT MULTI-AGENT SYSTEM
# ==========================

import pandas as pd
from sqlalchemy import create_engine, inspect
from openai import OpenAI

# ==========================
# CONFIG
# ==========================

DB_NAME = "hospitalmanagement"
DB_USER = "root"
DB_PASSWORD = "Root@123#"
DB_HOST = "localhost"

PATIENT_TABLE = "patient"
ADMISSION_TABLE = "appointment"
DEPARTMENT_TABLE = "department"

PATIENT_ID = "patientid"
DEPARTMENT_ID = "departmentid"
DEPARTMENT_NAME = "departmentname"
APPOINTMENT_ID = "appointmentid"

llm = OpenAI(api_key="YOUR_API_KEY")

# ==========================
# DB AGENT
# ==========================

def db_agent():
    print("[DB Agent] Connecting to database...")
    engine = create_engine(
        f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
    )
    print("[DB Agent] Connected")
    return engine

# ==========================
# SCHEMA AGENT
# ==========================

def schema_agent(engine):
    print("[Schema Agent] Inspecting schema...")
    inspector = inspect(engine)
    schema = {
        table: [col["name"] for col in inspector.get_columns(table)]
        for table in inspector.get_table_names()
    }
    return schema

# ==========================
# QUERY AGENT
# ==========================

def query_agent():
    print("\nAvailable Queries:")
    print("1. Calculate Patient Readmission Rates")
    choice = input("Select query number: ")
    return choice

# ==========================
# VALIDATION AGENT
# ==========================

def validation_agent(schema):
    required = [PATIENT_ID, APPOINTMENT_ID]
    missing = [
        col for col in required
        if col not in schema[ADMISSION_TABLE]
    ]
    return missing

# ==========================
# READMISSION LOGIC AGENT
# ==========================

def readmission_agent(engine):
    print("[Readmission Agent] Calculating admissions...")

    query = f"""
    SELECT 
        a.{PATIENT_ID},
        d.{DEPARTMENT_NAME},
        COUNT(a.{APPOINTMENT_ID}) AS total_admissions
    FROM {ADMISSION_TABLE} a
    JOIN {PATIENT_TABLE} p ON a.{PATIENT_ID} = p.{PATIENT_ID}
    JOIN {DEPARTMENT_TABLE} d ON p.{DEPARTMENT_ID} = d.{DEPARTMENT_ID}
    GROUP BY a.{PATIENT_ID}, d.{DEPARTMENT_NAME};
    """

    df = pd.read_sql(query, engine)

    df["readmissions"] = df["total_admissions"].apply(
        lambda x: x - 1 if x > 1 else 0
    )

    return df

# ==========================
# LLM EXPLANATION AGENT
# ==========================

def explanation_agent():
    prompt = """
Explain in simple terms how patient readmissions
can be calculated when there is no readmission column
in the database.
"""

    response = llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    print("\n[Explanation Agent]")
    print(response.choices[0].message.content)

# ==========================
# ORCHESTRATOR AGENT
# ==========================

def orchestrator():
    engine = db_agent()
    schema = schema_agent(engine)

    choice = query_agent()

    if choice == "1":
        missing = validation_agent(schema)

        if missing:
            print("\n[Validation Agent]")
            print("Missing column(s):", missing)
            print("Readmission will be DERIVED, not read from DB.")

        df = readmission_agent(engine)

        explanation_agent()

        print("\n[Output Agent] Top 5 Records:")
        print(df.head())

        print("\n[Department-wise Readmissions]")
        dept_summary = (
            df.groupby(DEPARTMENT_NAME)["readmissions"]
            .sum()
            .sort_values(ascending=False)
        )
        print(dept_summary)

# ==========================
# SYSTEM START
# ==========================

if __name__ == "__main__":
    orchestrator()


[DB Agent] Connecting to database...


ModuleNotFoundError: No module named 'pymysql'